# Data Parallel Loading: Simulating DistributedSampler — Solution

In [ ]:
import math
import torch
from torch.utils.data import Dataset, DataLoader, Sampler

class SimpleDataset(Dataset):
    def __init__(self, N):
        self.data = torch.arange(N, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = SimpleDataset(103)

In [ ]:
class SimpleDistributedSampler(Sampler):
    """Simplified DistributedSampler: shards a dataset across ranks."""

    def __init__(self, dataset, num_replicas: int, rank: int,
                 shuffle: bool = True, seed: int = 0):
        self.dataset = dataset
        self.num_replicas = num_replicas
        self.rank = rank
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.num_samples = math.ceil(len(dataset) / num_replicas)
        self.total_size = self.num_samples * num_replicas

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.num_samples

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(len(self.dataset), generator=g).tolist()
        else:
            indices = list(range(len(self.dataset)))

        # Pad to total_size by repeating from the start
        padding = self.total_size - len(indices)
        indices += indices[:padding]

        # Shard: this rank takes every num_replicas-th index
        indices = indices[self.rank::self.num_replicas]
        assert len(indices) == self.num_samples
        return iter(indices)


# Demo: show how indices are split across 4 ranks
for r in range(4):
    s = SimpleDistributedSampler(ds, num_replicas=4, rank=r, shuffle=False)
    idx = list(s)
    print(f"Rank {r}: {len(idx)} samples, first 5: {idx[:5]}, last 5: {idx[-5:]}")